# Lab 5 — NYSE Multi-Cluster View

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

Visual compare: partitioning clusters (K-Means) vs density clusters (DBSCAN).

**Dataset:** `data/nyse/nyse_stocks.csv` (500 rows, 25 symbols)

## Setup and dual-model inference

In [ ]:
%matplotlib inline

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

OUTPUT_DIR = GH_ROOT / "hands-on" / "05-unsupervised-learning" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])

kmeans_labels = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X_scaled)
dbscan_labels = DBSCAN(eps=1.2, min_samples=3).fit_predict(X_scaled)

features = features.copy()
features["kmeans"] = kmeans_labels
features["dbscan"] = dbscan_labels

print(f"symbols plotted: {len(features)}")


## Cluster counts from both methods

In [ ]:
kmeans_counts = dict(zip(*np.unique(kmeans_labels, return_counts=True)))
dbscan_counts = dict(zip(*np.unique(dbscan_labels, return_counts=True)))

print("Lab 5 — NYSE multi-cluster view")
print(f"K-Means cluster counts: {kmeans_counts}")
print(f"DBSCAN cluster counts: {dbscan_counts}")


## Side-by-side cluster plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, labels, title in zip(
    axes,
    [kmeans_labels, dbscan_labels],
    ["K-Means (k=4)", "DBSCAN"],
    strict=True,
):
    scatter = ax.scatter(
        features["avg_close"],
        features["volatility"],
        c=labels,
        cmap="tab10",
        alpha=0.85,
    )
    ax.set_xlabel("avg_close")
    ax.set_ylabel("volatility")
    ax.set_title(title)
    fig.colorbar(scatter, ax=ax, label="cluster")

fig.tight_layout()
plot_path = OUTPUT_DIR / "multi_cluster_view.png"
fig.savefig(plot_path, dpi=100)
plt.show()

print(f"plot saved: {plot_path.name}")
print(f"full path: {plot_path}")


## Inspect DBSCAN noise points

In [ ]:
noise = features.loc[features["dbscan"] == -1, ["symbol", "avg_close", "volatility"]]
display(noise.sort_values("volatility").round(2))
print(f"noise count: {len(noise)}")


## Agreement/disagreement table

In [ ]:
features["agree"] = features["kmeans"] == features["dbscan"]
disagree = features.loc[~features["agree"] | (features["dbscan"] == -1),
                      ["symbol", "kmeans", "dbscan"]]
display(disagree.sort_values("dbscan"))


## Render saved plot preview

In [ ]:
if plot_path.is_file():
    display(Image(filename=str(plot_path)))
else:
    print("Run the plot cell above first.")


### Visualization prompt 1

Show all symbols with both labels.

In [ ]:
display(features[['symbol','kmeans','dbscan']].sort_values(['kmeans','dbscan','symbol']))

### Visualization prompt 2

Compute simple agreement rate.

In [ ]:
print(round((features['kmeans']==features['dbscan']).mean(), 3))

### Visualization prompt 3

Agreement rate excluding noise.

In [ ]:
mask = features['dbscan']!=-1; print(round((features.loc[mask,'kmeans']==features.loc[mask,'dbscan']).mean(), 3))

### Visualization prompt 4

Top avg_close by K-Means cluster.

In [ ]:
display(features.sort_values('avg_close', ascending=False).groupby('kmeans').head(2)[['symbol','kmeans','avg_close']].round(2))

### Visualization prompt 5

Top volatility by DBSCAN label.

In [ ]:
display(features.sort_values('volatility', ascending=False).groupby('dbscan').head(2)[['symbol','dbscan','volatility']].round(3))

### Visualization prompt 6

Cross-tab of label overlap.

In [ ]:
display(pd.crosstab(features['kmeans'], features['dbscan']))

### Visualization prompt 7

List noise symbols only.

In [ ]:
print(features.loc[features['dbscan']==-1, 'symbol'].tolist())

### Visualization prompt 8

Check plot file metadata.

In [ ]:
print({'exists': plot_path.is_file(), 'size_bytes': plot_path.stat().st_size if plot_path.is_file() else 0})

### Visualization prompt 9

Summarize counts in one dict.

In [ ]:
print({'kmeans': kmeans_counts, 'dbscan': dbscan_counts})

### Visualization prompt 10

Mean feature values for disagreements.

In [ ]:
display(features.loc[(features['kmeans']!=features['dbscan']) | (features['dbscan']==-1)].groupby('dbscan')[FEATURE_COLUMNS].mean().round(2))

### Visualization prompt 11

Sort disagreement table by symbol.

In [ ]:
display(disagree.sort_values('symbol'))

### Visualization prompt 12

How many symbols in each K-Means cluster?

In [ ]:
print(features['kmeans'].value_counts().sort_index().to_dict())

### Visualization prompt 13

How many symbols in each DBSCAN label?

In [ ]:
print(features['dbscan'].value_counts().sort_index().to_dict())

### Visualization prompt 14

Plot compact agreement bar.

In [ ]:
pd.Series({'agree': int(features['agree'].sum()), 'disagree': int((~features['agree']).sum())}).plot(kind='bar', figsize=(4,3), title='Agreement count');

### Visualization prompt 15

Verify all 25 symbols are unique.

In [ ]:
print(features['symbol'].nunique(), len(features))

### Visualization prompt 16

Inspect symbols in singleton K-Means cluster 3.

In [ ]:
display(features.loc[features['kmeans']==3, ['symbol','avg_close','volatility']].round(2))

### Visualization prompt 17

Inspect DBSCAN label 2 members.

In [ ]:
display(features.loc[features['dbscan']==2, ['symbol','avg_close','volatility']].round(2))

### Visualization prompt 18

Bridge to segmentation summary.

In [ ]:
print('Next: convert cluster IDs into segment narratives and summaries.')

### Visualization prompt 19

Check cluster means by algorithm.

In [ ]:
display(features.groupby('kmeans')[FEATURE_COLUMNS].mean().round(2))

### Visualization prompt 20

Quick final visualization note.

In [ ]:
print('Keep the saved PNG for slide-ready cluster comparison.')

### Lab 5 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 5 recap step 1: completed")

### Lab 5 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 5 recap step 2: completed")

### Lab 5 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 5 recap step 3: completed")

### Lab 5 quick recap 4

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 5 recap step 4: completed")

## Final checkpoint

In [ ]:
assert len(features) == 25
assert plot_path.is_file()
assert kmeans_counts == {0: 8, 1: 9, 2: 7, 3: 1}
assert dbscan_counts[-1] == 11
assert dbscan_counts[0] == 6 and dbscan_counts[1] == 4 and dbscan_counts[2] == 4
print("Numbers match — you're good.")



## Reflection

When should teams present both partitioning and density views together?